In [2]:
# ============================================================
# Prerequisites
# ============================================================
import subprocess, sys, os

REQUIRED = ['requests', 'pandas']

def install(pkg):
    result = subprocess.run(['uv', 'add', pkg], capture_output=True, text=True, cwd=os.getcwd())
    if result.returncode == 0:
        return '✅', 'uv'
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg], capture_output=True, text=True)
    return ('✅', 'pip') if result.returncode == 0 else ('❌', 'failed')

for pkg in REQUIRED:
    status, method = install(pkg)
    print(f'  {status} {pkg} ({method})')
print('\n⚠️  Restart kernel after first install.')

  ✅ requests (uv)
  ✅ pandas (uv)

⚠️  Restart kernel after first install.


In [8]:
# ============================================================
# Setup
# ============================================================
import requests
import json
import pandas as pd
from IPython.display import display

POLARIS_URL = 'http://192.168.139.2:8181'
REALM       = 'POLARIS'

def get_token(client_id='root', client_secret='polaris-secret'):
    r = requests.post(
        f'{POLARIS_URL}/api/catalog/v1/oauth/tokens',
        headers={'Polaris-Realm': REALM},
        data={
            'grant_type': 'client_credentials',
            'client_id': client_id,
            'client_secret': client_secret,
            'scope': 'PRINCIPAL_ROLE:ALL'
        }
    )
    return r.json()['access_token']

def h(token):
    return {
        'Authorization': f'Bearer {token}',
        'Polaris-Realm': REALM,
        'Content-Type': 'application/json'
    }

TOKEN = get_token()
print('✅ Connected as root')

✅ Connected as root


In [9]:
# Grant PRINCIPAL_ROTATE_CREDENTIALS to service_admin
r = requests.put(
    f'{POLARIS_URL}/api/management/v1/principal-roles/service_admin/grants',
    headers=h(TOKEN),
    json={
        'grant': {
            'type': 'principal',
            'privilege': 'PRINCIPAL_ROTATE_CREDENTIALS'
        }
    }
)
print(f'Grant status: {r.status_code}')
if r.text:
    print(r.text)

Grant status: 404
{"error":{"message":"Unable to find matching target resource method","type":"NotFoundException","code":404}}


In [4]:
# ============================================================
# Step 1 — Find which principal has ops-admin role
# ============================================================
r = requests.get(
    f'{POLARIS_URL}/api/management/v1/principal-roles/ops-admin/principals',
    headers=h(TOKEN)
)
principals_with_ops_admin = r.json().get('principals', [])
print('Principals with ops-admin role:')
for p in principals_with_ops_admin:
    print(f'  → {p["name"]} (clientId: {p.get("clientId", "N/A")})')

Principals with ops-admin role:
  → alice (clientId: 892444e249cec8f9)


In [5]:
# ============================================================
# Step 2 — Rotate credentials for that principal
# ============================================================
TARGET = principals_with_ops_admin[0]['name']  # e.g. alice

r = requests.post(
    f'{POLARIS_URL}/api/management/v1/principals/{TARGET}/rotate',
    headers=h(TOKEN)
)
print(f'Status: {r.status_code}')
if r.status_code == 200:
    creds = r.json().get('credentials', {})
    CLIENT_ID     = creds.get('clientId')
    CLIENT_SECRET = creds.get('clientSecret')
    print(f'✅ New credentials for {TARGET}:')
    print(f'   clientId:     {CLIENT_ID}')
    print(f'   clientSecret: {CLIENT_SECRET}')
    print('\n⚠️  Save these — clientSecret is shown only once!')
else:
    print(f'❌ Failed: {r.text}')

Status: 403
❌ Failed: {"error":{"message":"Principal 'root' with activated PrincipalRoles '[service_admin]' and activated grants via '[service_admin]' is not authorized for op ROTATE_CREDENTIALS","type":"ForbiddenException","code":403}}


In [ ]:
# ============================================================
# Step 3 — Get catalog token using new credentials
# ============================================================
TOKEN_CATALOG = get_token(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET
)
print(f'✅ Catalog token obtained for {TARGET}')

# Verify — list namespaces on analytics-catalog
r = requests.get(
    f'{POLARIS_URL}/api/catalog/v1/analytics-catalog/namespaces',
    headers=h(TOKEN_CATALOG)
)
print(f'\nNamespaces in analytics-catalog:')
print(json.dumps(r.json(), indent=2))